In [1]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch import nn
import sklearn


In [2]:
df = pd.read_csv("test.csv")
df.head()
X, y = df["X"], df["y"]

X = torch.from_numpy(X.to_numpy()).type(dtype=torch.float32)
y = torch.from_numpy(y.to_numpy()).type(dtype=torch.float32)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [3]:
class Regularization_Term(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(1))
        self.bias = nn.Parameter(torch.rand(1))
    def forward(self,X):
        return self.weight*X+self.bias

In [4]:
def Acc_Func(y_true, y_pred):
    return (torch.eq(y_true, y_pred).sum().item() / len(y_true)) * 100

In [5]:
model30 = Regularization_Term()
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(params=model30.parameters(), lr=0.0001)
epochs = 2000

lambda_L2 = 0.01
lambda_L1 = 0.01
for epoch in range(epochs):
    model30.train()
    y_preds = model30(X_train)
    loss = (
        loss_fn(y_preds, y_train)
        + lambda_L1 * torch.sum(torch.abs(model30.weight))
        + lambda_L2 * torch.sum(model30.weight**2)
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model30.eval()
    with torch.inference_mode():
        test_preds = model30(X_test)
        test_loss = loss_fn(test_preds, y_test)

        # Round predictions to nearest integer
        y_pred_round = torch.round(test_preds)

        acc = Acc_Func(y_test, y_pred_round)
        if epoch % 20 == 0:
            print(
                f"Epoch: {epoch}, Loss: {loss:.6f}, Test Loss: {test_loss:.6f}, Accuracy: {acc:.2f}%"
            )

Epoch: 0, Loss: 9967.600586, Test Loss: 725.028320, Accuracy: 0.00%
Epoch: 20, Loss: 0.279221, Test Loss: 0.260179, Accuracy: 60.00%
Epoch: 40, Loss: 0.278805, Test Loss: 0.259685, Accuracy: 60.00%
Epoch: 60, Loss: 0.278392, Test Loss: 0.259193, Accuracy: 60.00%
Epoch: 80, Loss: 0.277976, Test Loss: 0.258699, Accuracy: 60.00%
Epoch: 100, Loss: 0.277563, Test Loss: 0.258208, Accuracy: 60.00%
Epoch: 120, Loss: 0.277149, Test Loss: 0.257715, Accuracy: 60.00%
Epoch: 140, Loss: 0.276738, Test Loss: 0.257228, Accuracy: 60.00%
Epoch: 160, Loss: 0.276327, Test Loss: 0.256738, Accuracy: 60.00%
Epoch: 180, Loss: 0.275918, Test Loss: 0.256253, Accuracy: 60.00%
Epoch: 200, Loss: 0.275507, Test Loss: 0.255766, Accuracy: 60.00%
Epoch: 220, Loss: 0.275098, Test Loss: 0.255279, Accuracy: 60.00%
Epoch: 240, Loss: 0.274689, Test Loss: 0.254792, Accuracy: 60.00%
Epoch: 260, Loss: 0.274282, Test Loss: 0.254309, Accuracy: 60.00%
Epoch: 280, Loss: 0.273878, Test Loss: 0.253828, Accuracy: 60.00%
Epoch: 300, 

In [6]:
print(model30.weight.item())
print(model30.bias.item())

2.012864112854004
0.1266815960407257


In [7]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test.numpy(),test_preds.numpy())
print(f"R2 Score: {r2:.4f}")

R2 Score: 0.9999


In [8]:
for name, param in model30.named_parameters():
    if param.grad is not None:
        print(name, param.grad)

weight tensor([0.0044])
bias tensor([-0.4152])


In [9]:
from sklearn.linear_model import Ridge

ridge_reg = Ridge(alpha=1, solver="cholesky")
ridge_reg.fit(X.reshape(-1, 1), y)

Ridge(alpha=1, solver='cholesky')

In [10]:
print(ridge_reg.predict([[2]]))
print(model30.weight.item()*2+model30.bias.item())

[5.0011641]
4.1524098217487335


In [11]:
from sklearn.linear_model import  ElasticNet
elastic_net = ElasticNet(alpha=1,l1_ratio=0.5)
elastic_net.fit(X.reshape(-1,1),y)



ElasticNet(alpha=1)

In [12]:
elastic_net.predict(X=[[2]])

array([5.08725637])

In [13]:
best_loss = float("inf")
patience = 5
counter = 0

In [14]:
for epoch in range(epochs):
    model30.train()
    y_preds = model30(X_train)

    loss = (
        loss_fn(y_preds, y_train)
        + lambda_L1 * torch.sum(torch.abs(model30.weight))
        + lambda_L2 * torch.sum(model30.weight**2)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model30.eval()
    with torch.inference_mode():
        test_preds = model30(X_test)
        test_loss = loss_fn(test_preds, y_test)

    if test_loss < best_loss:
        best_loss = test_loss
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break
    if epoch % 20 == 0:
        print(f"Epoch {epoch} | Train Loss {loss:.4f} | Test Loss {test_loss:.4f}")

Epoch 0 | Train Loss 0.2417 | Test Loss 0.2155
Epoch 20 | Train Loss 0.2413 | Test Loss 0.2151
Epoch 40 | Train Loss 0.2410 | Test Loss 0.2147
Epoch 60 | Train Loss 0.2406 | Test Loss 0.2143
Epoch 80 | Train Loss 0.2403 | Test Loss 0.2139
Epoch 100 | Train Loss 0.2399 | Test Loss 0.2135
Epoch 120 | Train Loss 0.2396 | Test Loss 0.2131
Epoch 140 | Train Loss 0.2393 | Test Loss 0.2127
Epoch 160 | Train Loss 0.2389 | Test Loss 0.2123
Epoch 180 | Train Loss 0.2386 | Test Loss 0.2119
Epoch 200 | Train Loss 0.2382 | Test Loss 0.2115
Epoch 220 | Train Loss 0.2379 | Test Loss 0.2111
Epoch 240 | Train Loss 0.2376 | Test Loss 0.2107
Epoch 260 | Train Loss 0.2372 | Test Loss 0.2103
Epoch 280 | Train Loss 0.2369 | Test Loss 0.2099
Epoch 300 | Train Loss 0.2366 | Test Loss 0.2095
Epoch 320 | Train Loss 0.2362 | Test Loss 0.2091
Epoch 340 | Train Loss 0.2359 | Test Loss 0.2087
Epoch 360 | Train Loss 0.2356 | Test Loss 0.2083
Epoch 380 | Train Loss 0.2352 | Test Loss 0.2079
Epoch 400 | Train Loss 0.2